# Import and predefining

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn import set_config
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch import nn
from torchvision import datasets, transforms

set_config(display="text")

random_state = 42

torch.manual_seed(random_state)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
def plot_heatmap(
    dataframe: pd.DataFrame | np.ndarray,
    figsize: tuple[int, int] = (6, 5),
    cmap: str | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    rotation: int = 45,
    xticklabels: list[str] | bool = True,
    yticklabels: list[str] | bool = True,
    title_name: str | None = None,
    ax: plt.Axes | None = None, # pyright: ignore[reportPrivateImportUsage]
) -> None:
    if ax is None:
        plt.figure(figsize=figsize)

    sns.heatmap(
        dataframe,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidths=0.5,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 7},
        xticklabels=xticklabels,
        yticklabels=yticklabels,
        ax=ax,
    )

    if ax is None:
        plt.xticks(rotation=rotation, ha="right", fontsize=9)
        plt.yticks(rotation=0, fontsize=9)
        if title_name:
            plt.title(title_name)
        plt.show()
    else:
        ax.tick_params(axis="x", rotation=rotation, labelsize=9)
        ax.tick_params(axis="y", rotation=0, labelsize=9)
        if title_name:
            ax.set_title(title_name)

def plot_table(
        cellText: np.ndarray | pd.DataFrame,
        rowLabels: list | None = None,
        colLabels: list | None = None,
        title_name: str | None = None,
        fontsize: int = 10,
        scale: tuple[float, float] = (1.7, 1.5),
        figsize: tuple[int, int] = (8, 3)
    ) -> None:
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')

    table = ax.table(
        cellText=cellText,
        rowLabels=rowLabels,
        colLabels=colLabels,
        cellLoc='center',
        loc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(*scale)
    if title_name:
        plt.title(title_name)
    plt.show()

# 1. Повнозв'язані нейронні мережі
1. Візьміть датасет, з яким ви працювали в лабораторній роботі №1.
1. Побудуйте повнозв'язану нейронну мережу (MLP) для задачі класифікації.
1. Навчіть модель на тренувальній вибірці та оцініть її якість на тестовій.
1. Побудуйте learning curves (train/validation loss) та проаналізуйте процес навчання моделі. Чи спостерігається перенавчання?
1. Порівняйте результати нейронної мережі з результатами класичних алгоритмів машинного навчання з Lab 1.
1. Зробіть висновки.

## Візьміть датасет, з яким ви працювали в лабораторній роботі №1

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

label_encoder_lab_1 = LabelEncoder()
scaler = StandardScaler()
datapath_lab_1 = Path("../lab1/data/Date_Fruit_Datasets.xlsx")
raw_data_lab_1 = pd.read_excel(datapath_lab_1)

target_name = "Class"
features_lab_1 = raw_data_lab_1.drop(target_name, axis=1)
target_lab_1 = raw_data_lab_1[target_name]

target_encoded_lab_1 = label_encoder_lab_1.fit_transform(target_lab_1)

X_train_lab_1, X_test_lab_1, y_train_lab_1, y_test_lab_1  = train_test_split(features_lab_1, target_encoded_lab_1, random_state=random_state, test_size=0.25, stratify=target_encoded_lab_1)

X_train_lab_1_scaled = scaler.fit_transform(X_train_lab_1)
X_test_lab_1_scaled = scaler.transform(X_test_lab_1)

In [5]:
X_train_lab_1_scaled.shape, X_test_lab_1_scaled.shape, y_train_lab_1.shape, y_test_lab_1.shape

((673, 34), (225, 34), (673,), (225,))

## Побудуйте повнозв'язану нейронну мережу (MLP) для задачі класифікації

In [6]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(34, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)   # logits
        )

    def forward(self, x):
        x = x.reshape(x.size(0), -1)  # [B, 34] -> [B, 34]
        logits = self.net(x)  # [B, 34] -> [B, 10]
        return logits  # [B, 10]

model = MLP().to(device)
print(model)

MLP(
  (net): Sequential(
    (0): Linear(in_features=34, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


## Навчіть модель на тренувальній вибірці та оцініть її якість на тестовій

In [7]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", num_params)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

patience = 2
best_val_loss = float('inf')
epochs_no_improve = 0
n_epochs = 50
best_model_path = "best_model.pth"

Trainable params: 13386


In [ ]:
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)

train_loss_history = []
val_loss_history = []
for epoch in range(n_epochs):

    # --- train ---
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)
    print(f"[{epoch+1:02d}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping epoch: ", epoch)
            break

RuntimeError: legacy constructor expects device type: cpu but device type: cuda was passed

## Побудуйте learning curves (train/validation loss) та проаналізуйте процес навчання моделі. Чи спостерігається перенавчання?

## Порівняйте результати нейронної мережі з результатами класичних алгоритмів машинного навчання з Lab 1

## Зробіть висновки

# 2. Згорткові нейронні мережі
1. Оберіть датасет зображень для задачі класифікації. 
1. Коротко опишіть датасет (класи, кількість зображень) та візуалізуйте приклади зображень для кожного класу. 
1. Вирішіть задачу класифікації двома способами.
    1. a)
        1. Побудуйте просту CNN (2–3 convolutional шари + fully connected layer).
        1. Навчіть модель на обраному датасеті.
        1. Побудуйте learning curves (train/validation loss) та проаналізуйте процес навчання.
    1. b) 
        1. Використайте попередньо натреновану модель (наприклад, ResNet, EfficientNet, MobileNet або іншу pretrained CNN).
        1. Замініть вихідний класифікатор відповідно до кількості класів вашого датасету та проведіть донавчання моделі.
1. Порівняйте CNN, навчену з нуля, та pretrained модель:  якість класифікації;  процес і швидкість збіжності;  learning curves;  вплив обсягу тренувальних даних.
1. Зробіть висновки.

## Оберіть датасет зображень для задачі класифікації

In [ ]:

datapath = Path("../lab3/data/")
random_state = 42

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_full = datasets.ImageFolder(datapath / "train", transform=train_transform)
val_full = datasets.ImageFolder(datapath / "train", transform=eval_transform)
test_dataset = datasets.ImageFolder(datapath / "test", transform=eval_transform)

labels_str = train_full.classes
targets = np.array(train_full.targets)
labels = np.unique(targets)

train_idx, val_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.25,
    random_state=random_state,
    stratify=targets,
)

train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(val_full, val_idx)

In [ ]:
from sklearn.datasets import load_files

label_encoder = LabelEncoder()
datapath = Path("../lab3/data/")

train_bundle = load_files(datapath / "train", load_content=False, shuffle=False)
test_bundle = load_files(datapath / "test", load_content=False, shuffle=False)

labels_str = np.unique(Y)
y = label_encoder.fit_transform(Y)
labels = np.unique(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=random_state,
    test_size=0.25,
    stratify=labels,
)

## Коротко опишіть датасет (класи, кількість зображень) та візуалізуйте приклади зображень для кожного класу

## Вирішіть задачу класифікації двома способами

### a)

#### Побудуйте просту CNN (2–3 convolutional шари + fully connected layer)

#### Навчіть модель на обраному датасеті

#### Побудуйте learning curves (train/validation loss) та проаналізуйте процес навчання

### b)

#### Використайте попередньо натреновану модель (наприклад, ResNet, EfficientNet, MobileNet або іншу pretrained CNN)

#### Замініть вихідний класифікатор відповідно до кількості класів вашого датасету та проведіть донавчання моделі

## Порівняйте CNN, навчену з нуля, та pretrained модель:  якість класифікації;  процес і швидкість збіжності;  learning curves;  вплив обсягу тренувальних даних

## Зробіть висновки

# 3. Нейронні мережі для класифікації текстів
1. Використайте той самий текстовий датасет, з яким ви працювали в лабораторній роботі №2.
1. Проведіть необхідну передобробку текстових даних: токенізацію, побудову словника, перетворення токенів в індекси та приведення послідовностей до однакової довжини (padding/truncation).
1. Вирішіть задачу класифікації двома способами.
1. 
    1. а)
        1. Побудуйте модель з Embedding layer, ініціалізованим випадковими вагами.
        1. Використайте RNN / LSTM / GRU для класифікації та навчіть модель на своєму датасеті.
    1. б)
        1. Завантажте готові pretrained embeddings (наприклад, GloVe) та ініціалізуйте ними Embedding layer.
        1. Проведіть навчання моделі.
1. Для обох моделей побудуйте learning curves (train/validation loss).
1. Порівняйте результати двох підходів:  якість класифікації;  швидкість і стабільність навчання;  вплив pretrained embeddings на отриманий результат.
1. Зробіть висновки.

## Використайте той самий текстовий датасет, з яким ви працювали в лабораторній роботі №2

In [ ]:
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

datapath_lab_2 = Path("../lab2/data/train.csv")
raw_data_lab_2 = pd.read_csv(datapath_lab_2)

raw_texts = raw_data_lab_2["text"]
targets = raw_data_lab_2["model"]
labels = np.sort(targets.unique())

raw_train, raw_test, targets_train, targets_test = train_test_split(
    raw_texts,
    targets,
    random_state=random_state,
    test_size=0.15,
    stratify=targets,
)

## Проведіть необхідну передобробку текстових даних: токенізацію, побудову словника, перетворення токенів в індекси та приведення послідовностей до однакової довжини (padding/truncation)

In [ ]:
def extract_style_features(texts: pd.Series) -> np.ndarray:
    feats = []
    for text in texts:
        length = max(len(text), 1)
        words = text.split()
        n_words = max(len(words), 1)
        sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
        n_sents = max(len(sentences), 1)

        feats.append([
            np.log1p(length),                                                 # total character length
            np.log1p(n_words),                                                # total word count
            length / n_words,                                                 # average word length
            n_words / n_sents,                                                # average sentence length (rhythm)
            text.count(",") / n_words,                                        # comma density (clause complexity)
            text.count(";") / n_words,                                        # semicolon usage
            text.count(":") / n_words,                                        # colon usage
            text.count("—") / n_words,                                        # em-dash usage
            text.count(" — ") / n_words,                                      # em-dash usage
            (text.count("’") + text.count("“") + text.count("”")) / n_words,  # curly quotes
            (text.count("'") + text.count('"')) / n_words,                    # straight quotes
            sum(c.isupper() for c in text) / length,                          # uppercase ratio
            sum(c.isdigit() for c in text) / length,                          # digit ratio
        ])
    return np.array(feats, dtype=np.float32)


vectorizer = FeatureUnion(
    transformer_list=[
        ("word", TfidfVectorizer(
            lowercase=False,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            token_pattern=r"\S+",
        )),
        ("char", TfidfVectorizer(
            lowercase=False,
            analyzer="char",
            ngram_range=(3, 5),
            min_df=5,
            max_df=0.95,
            sublinear_tf=True,
        )),
        ("style", Pipeline([
            ("extract", FunctionTransformer(extract_style_features, validate=False)),
            ("scale", StandardScaler()),
        ])),
    ], # type: ignore
    transformer_weights={"word": 1.0, "char": 1.0, "style": 0.25},
)

## Вирішіть задачу класифікації двома способами

### a)

#### Побудуйте модель з Embedding layer, ініціалізованим випадковими вагами

#### Використайте RNN / LSTM / GRU для класифікації та навчіть модель на своєму датасеті

In [ ]:
pipe_RNN = Pipeline([
    ("vec", vectorizer),
    ("clf", LogisticRegression(max_iter=1000, random_state=random_state)),
])

pipe_LSTM = Pipeline([
    ("vec", vectorizer),
    ("clf", LogisticRegression(max_iter=1000, random_state=random_state)),
])

pipe_GRU = Pipeline([
    ("vec", vectorizer),
    ("clf", LogisticRegression(max_iter=1000, random_state=random_state)),
])

NameError: name 'Pipeline' is not defined

### b)

#### Завантажте готові pretrained embeddings (наприклад, GloVe) та ініціалізуйте ними Embedding layer

#### Проведіть навчання моделі

## Для обох моделей побудуйте learning curves (train/validation loss)

## Порівняйте результати двох підходів:  якість класифікації;  швидкість і стабільність навчання;  вплив pretrained embeddings на отриманий результат

## Зробіть висновки